# Tome 1, Chapter 5: Primitive Roots (Python companion)

Exercises first, solutions at the end. Computer-science threads: **Diffie–Hellman key exchange**, and the
**baby-step giant-step** algorithm, which computes discrete logarithms in $O(\sqrt p)$ instead of $O(p)$.

In [ ]:
import math
import random
import time
import numpy as np
import matplotlib.pyplot as plt
from sympy import factorint, primerange, isprime, randprime, cyclotomic_poly, Symbol
# Reproducible teaching examples; these seeds are not for cryptographic keys.
import random
from sympy.core.random import seed as seed_sympy
random.seed(10105)
seed_sympy(20105)


## Before the experiment: An exponent clock

Modulo $17$, predict the order of $2$ before asking Python. Does being nonzero imply being primitive? This notebook uses **deterministic teaching randomness**. Its finite-field parameters and unauthenticated protocol are for arithmetic experiments, not for protecting messages.

**How to use this notebook.** Try the exercise cells first; they are intentionally blank. Then run the solutions in order in a fresh kernel. Assertions check exact identities or a stated finite range; plots and random trials supply evidence, not proofs.


## Exercises

**P5.1 (★) Finding primitive roots fast.** Show that $g$ is a primitive root modulo $p$ if and only if $g^{(p-1)/q} \not\equiv 1$ for every prime $q \mid p-1$. Use it to write `primitive_root(p)` and find the smallest primitive root of every prime below $10^5$. What is the largest one you meet?

In [ ]:
# Your code here


**P5.2 (★★) Artin's conjecture, numerically.** For the primes $p < 10^6$, compute the proportion for which $2$ is a primitive root, and compare with Artin's constant $A = \prod_q \left(1 - \frac{1}{q(q-1)}\right) \approx 0.3739558$ (product over primes). Same for $10$: these are the primes for which $1/p$ has the maximal period $p-1$.

In [ ]:
# Your code here


**P5.3 (★★) CS application: Diffie–Hellman.** Alice and Bob agree publicly on a prime $p$ and a primitive root $g$. Alice picks a secret $a$ and sends $A = g^a$, Bob picks $b$ and sends $B = g^b$; both compute $g^{ab} = B^a = A^b$. Simulate it with a 256-bit safe prime ($p = 2q+1$ with $q$ prime, where $g$ is primitive iff $g^2 \ne 1$ and $g^q \ne 1$).

In [ ]:
# Your code here


**P5.4 (★★) CS application: baby-step giant-step.** To solve $g^x \equiv h \pmod p$, write $x = im + j$ with $m = \lceil\sqrt{p-1}\rceil$: store all $g^j$ ($j < m$) in a dictionary, then compare with $h\,g^{-im}$ for $i = 0, 1, \dots$. Implement it, break Diffie–Hellman for a 40-bit prime, and time it against brute force for a 28-bit prime. Estimate the time and memory for baby-step giant-step with a 256-bit prime. Why does this estimate alone not establish cryptographic security?

In [ ]:
# Your code here


**P5.5 (★★★) Lifting to $p^2$ can fail.** Find the primes $p < 10^5$ for which the smallest primitive root $g$ modulo $p$ is **not** a primitive root modulo $p^2$ (equivalently $g^{p-1} \equiv 1 \pmod{p^2}$). Then check Exercise 5.7(b): every prime factor $q \nmid n$ of $\Phi_n(a)$ satisfies $q \equiv 1 \pmod n$, for $n \le 30$ and $a \le 10$.

In [ ]:
# Your code here


---
## Solutions

### P5.1

In [ ]:
def primitive_root(p):
    if p == 2:
        return 1
    qs = list(factorint(p - 1))
    for g in range(2, p):
        if all(pow(g, (p - 1) // q, p) != 1 for q in qs):
            return g

roots = {p: primitive_root(p) for p in primerange(2, 10**5)}
worst = max(roots, key=roots.get)
print(f"largest smallest primitive root below 10^5: g = {roots[worst]} for p = {worst}")

If $g$ is not primitive, its order is a proper divisor of $p-1$, hence divides $(p-1)/q$ for some prime $q$. Conversely if $g^{(p-1)/q} = 1$ the order is at most $(p-1)/q$. Smallest primitive roots stay tiny in practice. Under the generalised Riemann hypothesis they are $O((\log p)^6)$ (Shoup, 1990); unconditionally, the best known bounds are small powers of $p$.

### P5.2

In [ ]:
ps = list(primerange(3, 10**6))
def is_prim(g, p):
    return g % p != 0 and all(pow(g, (p - 1) // q, p) != 1 for q in factorint(p - 1))

prop2 = np.mean([is_prim(2, p) for p in ps])
prop10 = np.mean([is_prim(10, p) for p in ps if p != 5])
A = np.prod([1 - 1 / (q * (q - 1)) for q in primerange(2, 10**6)])
print(f"proportion with 2 primitive: {prop2:.5f}   with 10 primitive: {prop10:.5f}   Artin's constant: {A:.5f}")

### P5.3

In [ ]:
while True:
    q = randprime(2**254, 2**255)
    p = 2 * q + 1
    if isprime(p):
        break
g = next(g for g in range(2, p) if pow(g, 2, p) != 1 and pow(g, q, p) != 1)
a, b = random.randrange(2, p - 1), random.randrange(2, p - 1)
A_pub, B_pub = pow(g, a, p), pow(g, b, p)
assert pow(B_pub, a, p) == pow(A_pub, b, p)
print(f"safe prime of {p.bit_length()} bits, g = {g}; shared secret agreed:", hex(pow(B_pub, a, p))[:20] + "...")

### P5.4

In [ ]:
def bsgs(g, h, p):
    m = math.isqrt(p - 1) + 1
    table = {}
    e = 1
    for j in range(m):
        table.setdefault(e, j)
        e = e * g % p
    factor = pow(g, -m, p)
    gamma = h
    for i in range(m):
        if gamma in table:
            return i * m + table[gamma]
        gamma = gamma * factor % p

def brute(g, h, p):
    e = 1
    for x in range(p):
        if e == h:
            return x
        e = e * g % p

for bits, solvers in [(28, [brute, bsgs]), (40, [bsgs])]:
    p = randprime(2 ** (bits - 1), 2**bits)
    g = primitive_root(p)
    x = random.randrange(p - 1)
    h = pow(g, x, p)
    for solve in solvers:
        t = time.perf_counter()
        assert solve(g, h, p) == x
        print(f"{bits}-bit p, {solve.__name__}: {time.perf_counter() - t:.2f} s")

Baby-step giant-step needs $O(\sqrt p)$ time **and memory** in this full-group example: for a 256-bit prime that means about $2^{128}$ table entries. But finite-field discrete logarithms also admit **index-calculus and number-field-sieve attacks**, which exploit the representation of the group and are much faster than generic attacks. A 256-bit finite-field modulus is a teaching example, not a secure Diffie–Hellman parameter. Standard finite-field deployments use much larger moduli, suitable subgroups, validated parameters, and authenticated protocols. Safe primes avoid a smooth $p-1$, but that condition alone does not provide security.


### P5.5

In [ ]:
fails = [p for p, g in roots.items() if p > 2 and pow(g, p - 1, p * p) == 1]
print("primes p < 10^5 whose smallest primitive root fails modulo p^2:", fails)

X = Symbol("X")
for n in range(2, 31):
    Phi = cyclotomic_poly(n, X)
    for a in range(2, 11):
        for q in factorint(int(Phi.subs(X, a))):
            assert n % q == 0 or q % n == 1
print("every prime factor q of Phi_n(a) with q not dividing n is 1 mod n (n <= 30, a <= 10)")

## An exponent clock modulo 17

Join successive powers of 2 and 3 on the residue circle. Predict which path visits all sixteen nonzero residues. The geometry is a display of a finite group orbit, not a claim that multiplication is an ordinary plane rotation.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, g in zip(axes, [2, 3]):
    residues = [pow(g, k, 17) for k in range(17)]
    order = next(k for k in range(1, 17) if residues[k] == 1)
    theta = [2*math.pi*r/17 for r in residues[:order+1]]
    ax.plot(np.cos(theta), np.sin(theta), "o-", color="black", ms=4)
    for r in range(1, 17):
        t = 2*math.pi*r/17
        ax.text(1.16*math.cos(t), 1.16*math.sin(t), str(r), ha="center", va="center", fontsize=9)
    ax.set_title(f"g = {g}, order = {order}")
    ax.set_aspect("equal"); ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4); ax.axis("off")
plt.tight_layout(); plt.show()


## One more question before closing

Change the primitive root to its square modulo an odd prime. Which subgroup do you see? Does the baby-step giant-step table still have to contain p−1 distinct values?

Write one sentence identifying what was proved in the chapter, what was checked exactly by the code, and what remains a conjecture or a heuristic.


## Units modulo a power of two

Theorem: for $k\ge 3$ every unit modulo $2^k$ is uniquely $(-1)^a 5^b$ with $0\le b<2^{k-2}$, and it is a square if and only if it is $1$ modulo $8$.


In [ ]:
def v2(n):
    c = 0
    while n % 2 == 0:
        n //= 2
        c += 1
    return c

for m in range(0, 10):
    assert v2(pow(5, 2**m) - 1) == m + 2

for k in range(3, 12):
    mod = 2**k
    order = 2**(k - 2)
    powers = [pow(5, b, mod) for b in range(order)]
    assert len(set(powers)) == order
    assert pow(5, order, mod) == 1
    units = {(((-1)**a) * p) % mod for a in (0, 1) for p in powers}
    assert units == {n for n in range(mod) if n % 2 == 1}
    squares = {pow(u, 2, mod) for u in units}
    assert squares == {n for n in units if n % 8 == 1}

print('2-power unit checks passed for k=3..11')
